# 02 — Pipeline & Inferenz

Phase 3 (Pipeline bauen + erste Inferenz auf 12 Hand-Gold-Anzeigen), Phase 4 (Iterationen A + B — pro Iteration eigener Predictions-Dateiname und Run-Header-Update), Phase 6 (voller Korpus auf 7B + 3B-Kontrast auf euler).

Cheatsheets: `CHEATSHEETS/transformers-konzepte.md` (Modell, Chat-Template, JSON-Parsing), `CHEATSHEETS/gpu-zugang.md` (Spawn, GPU-Wahl, Memory).

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Modell | _ (z. B. `Qwen/Qwen2.5-7B-Instruct`) |
| Server | _ (gauss / euler) |
| GPU-Index | _ |
| Schema-Datei | `SCHEMA.md` |
| Aktueller Run-Tag | _ (`baseline` / `iter_A` / `iter_B` / `full_7b` / `full_3b`) |
| Predictions-Datei | _ (`predictions.jsonl` / `predictions_iter_A.jsonl` / …) |
| Truncation | _ Zeichen (initial ~2000) |

Bei jeder neuen Iteration: Run-Tag + Predictions-Datei + Datum aktualisieren.

## Phase 3 — Pipeline bauen + Baseline-Inferenz

### Block 3.1 — Modell laden + Pipeline-Skelett

Konventionen aus `CHEATSHEETS/transformers-konzepte.md`:
- `Qwen/Qwen2.5-7B-Instruct`, `torch_dtype=torch.float32`, `.to("cuda").eval()` (kein `device_map="auto"`)
- Chat-Template via `tokenizer.apply_chat_template(...)`
- `do_sample=False`, `max_new_tokens=200`, `pad_token_id=tokenizer.eos_token_id`
- Robustes JSON-Parsen mit Regex statt direktem `json.loads`
- Head-Truncation auf ~2000 Zeichen (V100/T100, 32 GB)

Predictions landen in `predictions.jsonl` (git-ignored). Eine Zeile pro Anzeige: `{"refnr": ..., <6 Schema-Felder>}`.

In [ ]:
import json
import re
import time
import warnings
from pathlib import Path

import pandas as pd
import torch
import transformers
from huggingface_hub.utils import disable_progress_bars
from transformers import AutoModelForCausalLM, AutoTokenizer

# Kosmetische Warnungen unterdrücken — Modell ist gecached, Widget-/Deprecation-Spam ist Noise
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")
transformers.logging.set_verbosity_error()
disable_progress_bars()

MODEL_NAME       = "Qwen/Qwen2.5-7B-Instruct"
KORPUS_PATH      = Path("../daten/eigener_korpus.jsonl")
GOLD_PATH        = Path("../annotation/meine_gold.csv")
PRED_PATH        = Path("predictions.jsonl")          # baseline
MAX_INPUT_CHARS  = 2000
MAX_NEW_TOKENS   = 200

# Falls dein Spawn alle 4 GPUs sieht und du genau eine willst:
# import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # VOR den torch-Imports

print(f"verfügbare GPUs: {torch.cuda.device_count()}")
print(f"aktive GPU    : {torch.cuda.current_device()} = {torch.cuda.get_device_name(0)}")

In [ ]:
# Modell + Tokenizer laden (V100/T100-Constraints: float32, kein device_map)
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to("cuda").eval()
print(f"Modell {MODEL_NAME} geladen in {time.time() - t0:.1f}s")
print(f"GPU-Speicher belegt: {torch.cuda.memory_allocated() / 1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# System-Prompt = Schema + ein konkretes Beispiel-JSON (stabilisiert Qwen-Output stark)

SYSTEM_PROMPT = """Du extrahierst strukturierte Informationen aus deutschen Stellenanzeigen.
Antworte AUSSCHLIESSLICH mit einem gültigen JSON-Objekt nach folgendem Schema — kein Begleittext, keine Markdown-Codefences.

Schema (alle 6 Felder zwingend vorhanden):
- "homeoffice": "ja" | "teilweise" | "nein" | "remote" | "nicht_genannt"
- "vertragsart": "ausbildung" | "festanstellung" | "praktikum" | "werkstudent" | "sonstiges"
- "erfahrungslevel": "junior" | "mid" | "senior" | "egal" | "nicht_genannt"
- "gehalt_min_eur": ganze Zahl (Untergrenze) ODER null
- "gehalt_zeitraum": "monat" | "jahr" | null  (null nur, wenn gehalt_min_eur null ist)
- "skills_top3": Array mit max. 3 technischen Skills aus dem Text (z. B. "Python", "SQL", "Power BI") — leeres Array, wenn keine genannt

Regeln:
- "nicht_genannt" NUR, wenn die Anzeige zum Feld wirklich nichts sagt — nicht als Sicherheits-Antwort bei Unsicherheit.
- Bei einer Range ("ab 50.000 €" / "50.000–60.000 €") nimm die untere Grenze als ganze Zahl ohne Tausender-Trennung.
- Ausbildungen sind immer "junior". Werkstudent/Praktikum → "praktikum"/"werkstudent" + erfahrungslevel meist "junior".
- skills_top3: nur konkrete Tools/Sprachen/Frameworks — KEINE Soft Skills ("Teamfähigkeit"), KEINE Sprachen ("Englisch"), KEINE Fachgebiete ("Informatik").

Beispiel-Output:
{"homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "mid", "gehalt_min_eur": 55000, "gehalt_zeitraum": "jahr", "skills_top3": ["Python", "SQL", "Power BI"]}"""


def baue_messages(text: str) -> list[dict]:
    """System + User-Turn. Anzeige wird auf MAX_INPUT_CHARS Zeichen head-truncated."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Stellenanzeige:\n\n{text[:MAX_INPUT_CHARS]}"},
    ]


def extrahiere_json(response: str) -> dict | None:
    """Zieht das erste JSON-Objekt aus dem Modell-Output. Returns None bei Parse-Fail."""
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def inferenz(text: str) -> tuple[dict | None, str]:
    """Pro Anzeige: prompt → generate → JSON-Parse. Returns (parsed_dict_or_None, raw_response)."""
    messages = baue_messages(text)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    return extrahiere_json(response), response

### Block 3.2 — Inferenz auf 12 Hand-Gold-Anzeigen

Wir filtern den Korpus auf die 12 IDs aus `meine_gold.csv` und schreiben pro Anzeige eine JSONL-Zeile. Erwartete Laufzeit auf V100/T100 (7B fp32): ~40–60 s.

In [ ]:
# 12 Hand-Gold-IDs aus meine_gold.csv laden und Korpus filtern
gold_df = pd.read_csv(GOLD_PATH)
gold_ids = gold_df["id"].astype(str).tolist()

korpus = pd.read_json(KORPUS_PATH, lines=True)
anzeigen = korpus[korpus["refnr"].isin(gold_ids)].copy()
anzeigen = anzeigen.set_index("refnr").loc[gold_ids].reset_index()   # gleiche Reihenfolge wie gold

print(f"Hand-Gold-IDs : {len(gold_ids)}")
print(f"im Korpus     : {len(anzeigen)}")
assert len(anzeigen) == len(gold_ids), "Fehlende refnr — meine_gold.csv vs. eigener_korpus.jsonl vergleichen"

In [ ]:
# Inferenz-Schleife → predictions.jsonl
predictions  = []
parse_fails  = 0
t_start      = time.time()

with PRED_PATH.open("w", encoding="utf-8") as f:
    for i, row in enumerate(anzeigen.itertuples(index=False), 1):
        t_i = time.time()
        parsed, raw = inferenz(row.text)
        dt = time.time() - t_i

        if parsed is None:
            parse_fails += 1
            entry = {"refnr": row.refnr, "_parse_fail": True, "_raw": raw[:400]}
            tag = "PARSE_FAIL"
        else:
            entry = {"refnr": row.refnr, **parsed}
            tag = "OK"

        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        predictions.append(entry)
        print(f"  [{i:02d}/{len(anzeigen)}] {row.refnr}  {tag:<10}  ({dt:.1f}s)")

print(f"\nFertig: {len(predictions)} Anzeigen in {time.time() - t_start:.0f}s")
print(f"JSON-Parse-Fails: {parse_fails}/{len(predictions)}")
print(f"Predictions → {PRED_PATH.resolve()}")

## Phase 4 — Iteration A

## Phase 4 — Iteration B

## Phase 6 — Voller 7B-Run (gauss)

## Phase 6 — 3B-Run (euler)